In [ ]:
# ============================================================
# ASSIGNMENT 2 - Q2 TWO-STAGE RETRIEVE-THEN-RANK (MIND, LARGE) + Q4 serving data
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# STAGE 1 (retrieval): FAISS ANN over the FULL 104k article pool.
#   - content retrieval (MiniLM embeddings) + CTR-based candidate augmentation
#     (CTR is MIND-appropriate; recency is NOT used - MIND lacks reliable timestamps)
#   - compare FAISS Flat (exact) vs HNSW (approximate) : recall@K, p99 latency, memory
# STAGE 2 (rerank): score retrieved top-K with THREE best rerankers:
#   - LightGBM (best GBDT), MLP (best feature-neural), NRMS title+abstract (best text-neural)
# METRICS: retrieval recall@K ; conditional MRR/nDCG (clicked article was retrieved) ;
#          end-to-end MRR/nDCG (non-retrieved click = 0) ; extended eval + bootstrap CIs.
# Eval on a 20k dev-impression sample (metrics converge; pool+models are full-scale).
#
# HF donbosoc/mind-artifacts: download WITHOUT token; (no writes needed here).
# NEEDS: Kaggle MIND-large mounted.
# ============================================================
!pip install faiss-cpu lightgbm huggingface_hub sentence-transformers -q
import os, glob, re, math, numpy as np, pandas as pd, datetime as dt, random, time
from bisect import bisect_left
from collections import defaultdict, Counter
import faiss, torch, torch.nn as nn
from huggingface_hub import hf_hub_download
random.seed(0); torch.manual_seed(0)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:",device)
HF_REPO="donbosoc/mind-artifacts"
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")   # public read

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
EVAL_N=20000; Ks=[100,200]; MAX_LEN=50; MAX_HIST=30


In [ ]:
# ---- article catalogue + MiniLM embeddings (the retrieval pool) ----
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","category","title","abstract"]) for d in (LTR,LDV)]).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna(""); news["category"]=news["category"].fillna("")
ids=[pfx(r.news_id) for r in news.itertuples()]
id_to_row={x:i for i,x in enumerate(ids)}
cat_lut={pfx(r.news_id):r.category for r in news.itertuples()}
title_tok={pfx(r.news_id):tok(r.title) for r in news.itertuples()}
text_tokens={pfx(r.news_id):tok(f"{r.title} {r.abstract}")[:MAX_LEN] for r in news.itertuples()}

from sentence_transformers import SentenceTransformer
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb=minilm.encode([f"{r.title} {r.abstract}".strip() for r in news.itertuples()],
                  batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True).astype("float32")
emb_by_id={ids[i]:emb[i] for i in range(len(ids))}
print("pool size:", len(ids), "| emb:", emb.shape)


In [ ]:
# ---- point-in-time events (features + CTR augmentation), leakage-safe (exclude same iid) ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
click_ev=defaultdict(list); imp_ev=defaultdict(list); first_seen={}
for b in (b_tr,b_dv):
    for t,imps in zip(b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tkn in imps.split():
            p=tkn.split("-"); nid=pfx(p[0])
            if nid not in first_seen or t<first_seen[nid]: first_seen[nid]=t
            if len(p)==2:
                imp_ev[nid].append(t)
                if p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T); return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w); s=cnt(imp_ev,aid,T,w); return c/s if s>0 else 0.0
def freshness(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def user_cat(uid,mh=50):
    ai=hist_lut.get(uid,[])[-mh:]; cats=[cat_lut.get(a) for a in ai]
    tot=len([c for c in cats if c]); cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
# global CTR table (point-in-time not needed for the AUGMENTATION pool ranking; we use a
# recent-window CTR snapshot to pick high-CTR articles as extra candidates)
print("event index ready")


In [ ]:
# ---- STAGE 1: build FAISS indexes (Flat exact vs HNSW approx) + measure memory/latency ----
d=emb.shape[1]
# Flat (exact inner product)
t0=time.time(); flat=faiss.IndexFlatIP(d); flat.add(emb); flat_build=time.time()-t0
# HNSW (approximate)
t0=time.time(); hnsw=faiss.IndexHNSWFlat(d, 32); hnsw.hnsw.efConstruction=200
hnsw.add(emb); hnsw_build=time.time()-t0; hnsw.hnsw.efSearch=128
import sys as _sys
def idx_mem(ix):  # rough serialized size
    faiss.write_index(ix,"/kaggle/working/_tmp.index"); return os.path.getsize("/kaggle/working/_tmp.index")/1e6
print(f"Flat: build {flat_build:.1f}s, mem {idx_mem(flat):.1f} MB")
print(f"HNSW: build {hnsw_build:.1f}s, mem {idx_mem(hnsw):.1f} MB")

# latency probe: 1000 random queries, p50/p99
q=emb[np.random.default_rng(0).integers(0,len(emb),1000)]
for name,ix in [("Flat",flat),("HNSW",hnsw)]:
    lat=[]
    for i in range(len(q)):
        t0=time.time(); ix.search(q[i:i+1],200); lat.append((time.time()-t0)*1000)
    lat=np.array(lat)
    print(f"{name} search latency: p50 {np.percentile(lat,50):.2f}ms  p99 {np.percentile(lat,99):.2f}ms")


In [ ]:
# ---- eval set: 20k dev impressions with a clicked article + history ----
dev=[]
for iid,u,t,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["t"],b_dv["impressions"]):
    if not isinstance(imps,str) or pd.isna(t): continue
    uid=pfx(u)
    if not hist_lut.get(uid): continue
    clicked=[pfx(x.split("-")[0]) for x in imps.split() if x.endswith("-1")]
    if clicked: dev.append((iid,uid,t,clicked[0]))
random.Random(0).shuffle(dev); dev=dev[:EVAL_N]
print("eval impressions:", len(dev))

def user_vec(uid):
    hv=[emb_by_id[a] for a in hist_lut.get(uid,[])[-MAX_HIST:] if a in emb_by_id]
    if not hv: return None,None
    um=np.mean(hv,0); um=um/(np.linalg.norm(um)+1e-9); return um.astype("float32"), hv

# CTR-augmentation pool: top high-CTR articles (global snapshot) to add as candidates
ctr_snapshot={a: (len(click_ev.get(a,[]))/(len(imp_ev.get(a,[]))+1)) for a in ids}
top_ctr=[a for a,_ in sorted(ctr_snapshot.items(), key=lambda z:-z[1])[:500]]
top_ctr_rows=[id_to_row[a] for a in top_ctr]
print("CTR-augmentation pool:", len(top_ctr))


In [ ]:
# ---- STAGE 1 retrieval: FAISS top-K (+CTR augmentation) ; recall@K ----
def retrieve(um, K, index, augment=True):
    _,I=index.search(um.reshape(1,-1), K)
    cand=list(I[0])
    if augment:
        for r in top_ctr_rows:
            if r not in cand: cand.append(r)
            if len(cand)>=K+200: break
    return cand[:K+ (200 if augment else 0)]

# measure recall@K for content-only vs content+CTR, using HNSW (the ANN)
for augment in (False, True):
    hits={k:0 for k in Ks}; n=0
    for iid,uid,T,clicked in dev:
        um,hv=user_vec(uid)
        if um is None: continue
        crow=id_to_row.get(clicked)
        if crow is None: continue
        n+=1
        cand=retrieve(um, max(Ks), hnsw, augment=augment)
        for k in Ks:
            if crow in set(cand[:k] + (top_ctr_rows[:200] if augment else [])[:0]) or crow in cand[:k]:
                hits[k]+=1
    tag="content+CTR" if augment else "content-only"
    print(f"[{tag}] " + " ".join(f"recall@{k}={hits[k]/n:.4f}" for k in Ks) + f"  (n={n})")
print("\n(content retrieval is weak on MIND ~0.03-0.05 per A1; CTR augmentation lifts recall)")


In [ ]:
# ---- load the THREE best rerankers ----
import lightgbm as lgb
FEAT=["hist_emb_mean","hist_emb_best","recency_wt_sim","click_count","pop_1h","pop_24h",
      "pop_7d","pop_total","ctr_24h","ctr_total","freshness","cat_match","position","slate_size"]
lgbm=lgb.Booster(model_file=dl("mind_lgbm_reranker.txt"))
print("loaded LightGBM")

# MLP
class MLP(nn.Module):
    def __init__(s,din):
        super().__init__(); s.net=nn.Sequential(nn.Linear(din,128),nn.ReLU(),nn.Dropout(0.2),
                                                 nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(s,x): return s.net(x).squeeze(-1)
mlp=MLP(len(FEAT)).to(device); mlp.load_state_dict(torch.load(dl("mind_mlp.pt"),map_location=device)); mlp.eval()
print("loaded MLP")

# NRMS title+abstract
class AdditiveAttention(nn.Module):
    def __init__(s,dim,hid=200):
        super().__init__(); s.p=nn.Linear(dim,hid); s.q=nn.Linear(hid,1,bias=False)
    def forward(s,x,mask=None):
        e=torch.tanh(s.p(x)); a=s.q(e).squeeze(-1)
        if mask is not None: a=a.masked_fill(mask==0,-1e9)
        return (x*torch.softmax(a,-1).unsqueeze(-1)).sum(1)
class NewsEncoder(nn.Module):
    def __init__(s,V,ed=300,h=15,hd=20):
        super().__init__(); dd=h*hd
        s.emb=nn.Embedding(V,ed,padding_idx=0); s.attn=nn.MultiheadAttention(ed,h,batch_first=True)
        s.proj=nn.Linear(ed,dd); s.add=AdditiveAttention(dd); s.drop=nn.Dropout(0.2)
    def forward(s,t):
        m=(t!=0); x=s.drop(s.emb(t)); x,_=s.attn(x,x,x,key_padding_mask=~m)
        x=s.drop(x); x=torch.relu(s.proj(x)); return s.add(x,m)
class UserEncoder(nn.Module):
    def __init__(s,dd):
        super().__init__(); s.attn=nn.MultiheadAttention(dd,5,batch_first=True); s.add=AdditiveAttention(dd)
    def forward(s,hv,mask):
        x,_=s.attn(hv,hv,hv,key_padding_mask=~mask); return s.add(x,mask)
class NRMS(nn.Module):
    def __init__(s,V,h=15,hd=20):
        super().__init__(); s.news=NewsEncoder(V,h=h,hd=hd); s.user=UserEncoder(h*hd)
ck=torch.load(dl("mind_nrms_ta_best.pt"),map_location=device)
nrms_vocab=ck["vocab"]
nrms=NRMS(len(nrms_vocab)).to(device); nrms.load_state_dict(ck["model"]); nrms.eval()
def nrms_enc(aid):
    idx=[nrms_vocab.get(w,1) for w in text_tokens.get(aid,[])][:MAX_LEN]
    return idx+[0]*(MAX_LEN-len(idx))
nrms_text_idx={a:nrms_enc(a) for a in ids}
print("loaded NRMS(title+abstract)")


In [ ]:
# ---- feature builder for retrieved candidates (LightGBM/MLP inputs) ----
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
def recency_wt_hist(uid,mh=50,tau=8.0):
    ai=hist_lut.get(uid,[])[-mh:]; vs=[emb_by_id[a] for a in ai if a in emb_by_id]
    if not vs: return None
    nn_=len(vs); w=np.exp(-(np.arange(nn_)[::-1])/tau); w/=w.sum()
    um=np.average(np.array(vs),axis=0,weights=w); return um/(np.linalg.norm(um)+1e-9)
def build_feats(uid,T,cand_ids):
    hv=[emb_by_id[a] for a in hist_lut.get(uid,[])[-50:] if a in emb_by_id]
    um=(np.mean(hv,0)/(np.linalg.norm(np.mean(hv,0))+1e-9)) if hv else None
    rw=recency_wt_hist(uid); uc=user_cat(uid); hlen=len(hist_lut.get(uid,[])); m=len(cand_ids); F=[]
    for i,c in enumerate(cand_ids):
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        rws=float(rw@cv) if (rw is not None and cv is not None) else 0.0
        p1=cnt(click_ev,c,T,1);p24=cnt(click_ev,c,T,24);p7=cnt(click_ev,c,T,168);ptot=cnt(click_ev,c,T)
        c24=ctr(c,T,24);ctot=ctr(c,T);fr=freshness(c,T);cm=uc.get(cat_lut.get(c,""),0.0);pos=i/max(1,m-1)
        F.append([em,eb,rws,hlen,p1,p24,p7,ptot,c24,ctot,fr,cm,pos,m])
    F=np.array(F,float)
    for col in [4,5,6,7]: F[:,col]=mm(F[:,col])
    return F
print("feature builder ready")


In [ ]:
# ---- STAGE 2 rerank + metrics for all three models (end-to-end retrieve-then-rank) ----
def _mrr_rank(scores, cand_rows, clicked_row):
    # rank of clicked article among reranked candidates; 0 if not retrieved
    if clicked_row not in cand_rows: return 0.0
    order=np.argsort(-scores); ranked=[cand_rows[i] for i in order]
    r=ranked.index(clicked_row)+1; return 1.0/r
def _ndcg_at(scores, cand_rows, clicked_row, k):
    if clicked_row not in cand_rows: return 0.0
    order=np.argsort(-scores)[:k]; ranked=[cand_rows[i] for i in order]
    return 1.0/np.log2(ranked.index(clicked_row)+2) if clicked_row in ranked else 0.0

K=200
results={m:{"mrr_e2e":[],"ndcg10_e2e":[],"mrr_cond":[],"ndcg10_cond":[]} for m in ("LightGBM","MLP","NRMS")}
n=0; caught=0
for iid,uid,T,clicked in dev:
    um,hv=user_vec(uid)
    if um is None: continue
    crow=id_to_row.get(clicked)
    if crow is None: continue
    n+=1
    cand_rows=retrieve(um, K, hnsw, augment=True)
    cand_ids=[ids[r] for r in cand_rows]
    is_caught = crow in cand_rows
    if is_caught: caught+=1
    # features for LightGBM/MLP
    F=build_feats(uid,T,cand_ids)
    lgb_s=lgbm.predict(F)
    with torch.no_grad():
        mlp_s=mlp(torch.tensor(F,dtype=torch.float32).to(device)).cpu().numpy()
    # NRMS scores
    with torch.no_grad():
        hr=[nrms_text_idx.get(a,[0]*MAX_LEN) for a in hist_lut.get(uid,[])[-MAX_HIST:]]
        hrp=hr+[[0]*MAX_LEN]*(MAX_HIST-len(hr)); hmask=[1]*len(hr)+[0]*(MAX_HIST-len(hr))
        ht=torch.tensor([hrp]).to(device); hm=torch.tensor([hmask],dtype=torch.bool).to(device)
        ct=torch.tensor([[nrms_text_idx.get(a,[0]*MAX_LEN) for a in cand_ids]]).to(device)
        hv2=nrms.news(ht.reshape(-1,MAX_LEN)).reshape(1,MAX_HIST,-1); u=nrms.user(hv2,hm)
        cv=nrms.news(ct.reshape(-1,MAX_LEN)).reshape(1,len(cand_ids),-1)
        nrms_s=torch.bmm(cv,u.unsqueeze(-1)).squeeze(0).squeeze(-1).cpu().numpy()
    for name,sc in [("LightGBM",lgb_s),("MLP",mlp_s),("NRMS",nrms_s)]:
        results[name]["mrr_e2e"].append(_mrr_rank(sc,cand_rows,crow))
        results[name]["ndcg10_e2e"].append(_ndcg_at(sc,cand_rows,crow,10))
        if is_caught:
            results[name]["mrr_cond"].append(_mrr_rank(sc,cand_rows,crow))
            results[name]["ndcg10_cond"].append(_ndcg_at(sc,cand_rows,crow,10))
print(f"eval n={n} | retrieval caught the click in {caught} ({caught/n:.4f})\n")
print("=== TWO-STAGE retrieve-then-rank (HNSW retrieve K=200 + CTR aug -> rerank) ===")
print(f"{'model':10s} {'MRR(e2e)':>9} {'nDCG@10(e2e)':>13} {'MRR(cond)':>10} {'nDCG@10(cond)':>14}")
for m in ("LightGBM","MLP","NRMS"):
    r=results[m]
    print(f"{m:10s} {np.mean(r['mrr_e2e']):>9.4f} {np.mean(r['ndcg10_e2e']):>13.4f} "
          f"{np.mean(r['mrr_cond']):>10.4f} {np.mean(r['ndcg10_cond']):>14.4f}")
print("\ne2e = over all impressions (non-retrieved click counts 0) = true two-stage system")
print("cond = only where retrieval caught the click = reranker quality in isolation")


In [ ]:
# ---- bootstrap 95% CI on end-to-end MRR for each reranker ----
def bci(vals,nb=500):
    v=np.array(vals); rng=np.random.default_rng(0)
    m=[rng.choice(v,len(v),replace=True).mean() for _ in range(nb)]
    return v.mean(),np.percentile(m,2.5),np.percentile(m,97.5)
print("=== end-to-end MRR with bootstrap 95% CI ===")
for m in ("LightGBM","MLP","NRMS"):
    mean,lo,hi=bci(results[m]["mrr_e2e"])
    print(f"{m:10s} MRR {mean:.4f} [{lo:.4f}, {hi:.4f}]")
print("\nTwo-stage pipeline complete: FAISS ANN retrieval (full 104k pool) -> 3 rerankers -> metrics.")
print("FAISS Flat-vs-HNSW latency/memory (earlier cell) = Q4 serving/scale data.")
